In [46]:
import requests
import pandas as pd
import time
from datetime import datetime
import os

In [16]:
app_ids = [
    730,      # Counter-Strike 2
    570,      # Dota 2
    440,      # Team Fortress 2
    578080,   # PUBG
    271590,   # GTA V
    1172470,  # Apex Legends
    1085660,  # Destiny 2
    381210,   # Dead by Daylight
    1091500,  # Cyberpunk 2077
    1245620,  # ELDEN RING
    1938090,  # Call of Duty
    252490,   # Rust
    1599340,  # Lost Ark
    413150,   # Stardew Valley
    814380,   # Sekiro
    292030,   # The Witcher 3
    553850,   # HELLDIVERS 2
    236390,   # War Thunder
    394360,   # Hearts of Iron IV
    1086940   # Baldur's Gate 3
]

In [27]:
url = f"https://api.steampowered.com/ISteamNews/GetNewsForApp/v2/?appid={730}"
response = requests.get(url)
data = response.json()

print(data.keys())

dict_keys(['appnews'])


In [18]:
data["appnews"]['newsitems']

[{'gid': '1830163047268843',
  'title': '"Not all machines are twisted mechano-socialists": Helldivers 2\'s latest warbond brings back a mech suit from the original game',
  'url': 'https://steamstore-a.akamaihd.net/news/externalpost/Rock, Paper, Shotgun/1830163047268843',
  'is_external_url': True,
  'author': '',
  'contents': '<img src="https://assetsio.gnwcdn.com/helldivers-2-exo-experts-warbond-01.jpg?width=690&quality=85&format=jpg&auto=webp" /> <p><a href="https://www.rockpapershotgun.com/games/helldivers-2">Helldivers 2</a>\'s latest warbond is set to deliver two new exosuit stratagems, bringing a bit more variety to the robotic getups you can summon to help fight Super Earth\'s enemy of the week. One\'s the Helldivers 2 debut of a mech from the first game, while the other\'s got a big shield. Unsurprisingly, the gear pack they\'re part of is dubbed <a href="https://store.steampowered.com/news/app/553850/view/623327006824596290?l=english">Exo Experts</a>, and it\'s set to arriv

In [19]:
news_items = data["appnews"]["newsitems"]

unique_tags = set()

for post in news_items:
    tags = post.get("tags", [])
    for tag in tags:
        unique_tags.add(tag)

print(unique_tags)

{'ModAct_1425147233_1773748516_4', 'ModAct_1415716708_1774346912_4', 'ModAct_1415716708_1773743165_0', 'mod_require_rereview', 'patchnotes', 'mod_reviewed'}


In [20]:
data["appnews"]["newsitems"][0]

{'gid': '1830163047268843',
 'title': '"Not all machines are twisted mechano-socialists": Helldivers 2\'s latest warbond brings back a mech suit from the original game',
 'url': 'https://steamstore-a.akamaihd.net/news/externalpost/Rock, Paper, Shotgun/1830163047268843',
 'is_external_url': True,
 'author': '',
 'contents': '<img src="https://assetsio.gnwcdn.com/helldivers-2-exo-experts-warbond-01.jpg?width=690&quality=85&format=jpg&auto=webp" /> <p><a href="https://www.rockpapershotgun.com/games/helldivers-2">Helldivers 2</a>\'s latest warbond is set to deliver two new exosuit stratagems, bringing a bit more variety to the robotic getups you can summon to help fight Super Earth\'s enemy of the week. One\'s the Helldivers 2 debut of a mech from the first game, while the other\'s got a big shield. Unsurprisingly, the gear pack they\'re part of is dubbed <a href="https://store.steampowered.com/news/app/553850/view/623327006824596290?l=english">Exo Experts</a>, and it\'s set to arrive next

In [21]:
readable_date = datetime.fromtimestamp(data["appnews"]["newsitems"][0]['date'])

print(readable_date)
print((data["appnews"]["newsitems"][0]['feed_type']))
print((data["appnews"]["newsitems"][0]['feedlabel']))


2026-04-21 10:16:49
0
Rock, Paper, Shotgun


In [28]:

len(data["appnews"]["newsitems"])

20

In [40]:
def get_content_support(app_id):
    url = f"https://api.steampowered.com/ISteamNews/GetNewsForApp/v2/?appid={app_id}"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()

        news_items = data.get("appnews", {}).get("newsitems", [])
        if not news_items:
            return {
                "app_id": app_id,
                "latest_date": None,
                "feed_type": None,
                "feedlabel": None,
                "tags": None
            }

        latest_post = news_items[0]
        chosen_post = latest_post

        for post in news_items:
            if "patchnotes" in post.get("tags", []):
                chosen_post = post
                break

        return {
            "app_id": app_id,
            "latest_date": datetime.fromtimestamp(chosen_post["date"]),
            "feed_type": chosen_post.get("feed_type"),
            "feedlabel": chosen_post.get("feedlabel"),
            "tags": ", ".join(chosen_post.get("tags", []))
        }

    except Exception as e:
        print(f"Error for app_id {app_id}: {e}")
        return None

In [41]:
content_support = []

for app_id in app_ids:
    row = get_content_support(app_id)
    content_support.append(row)
    time.sleep(1)

df_content = pd.DataFrame(content_support)
df_content.head()

,app_id,latest_date,feed_type,feedlabel,tags
0,730,2026-04-22 18:40:51,1,Community Announcements,patchnotes
1,570,2026-04-07 16:24:46,1,Community Announcements,patchnotes
2,440,2026-03-11 15:33:26,1,Community Announcements,patchnotes
3,578080,2026-04-23 14:23:44,1,Community Announcements,
4,271590,2026-04-17 10:40:18,1,Community Announcements,hide_store


In [42]:
print(df_content.shape)
print(df_content.info())

print(df_content.isnull().sum())


(20, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   app_id       20 non-null     int64         
 1   latest_date  20 non-null     datetime64[ns]
 2   feed_type    20 non-null     int64         
 3   feedlabel    20 non-null     object        
 4   tags         20 non-null     object        
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 932.0+ bytes
None
app_id         0
latest_date    0
feed_type      0
feedlabel      0
tags           0
dtype: int64


In [45]:
print(df_content[df_content['tags'] == 'patchnotes'])

     app_id         latest_date  feed_type                feedlabel  \
0       730 2026-04-22 18:40:51          1  Community Announcements   
1       570 2026-04-07 16:24:46          1  Community Announcements   
2       440 2026-03-11 15:33:26          1  Community Announcements   
5   1172470 2026-04-07 17:10:40          1  Community Announcements   
9   1245620 2025-12-16 02:05:57          1  Community Announcements   
16   553850 2026-03-25 06:00:50          1  Community Announcements   
17   236390 2026-04-23 07:47:15          1  Community Announcements   
18   394360 2026-03-26 11:03:39          1  Community Announcements   
19  1086940 2026-03-26 10:40:05          1  Community Announcements   

          tags  
0   patchnotes  
1   patchnotes  
2   patchnotes  
5   patchnotes  
9   patchnotes  
16  patchnotes  
17  patchnotes  
18  patchnotes  
19  patchnotes  


In [44]:
df_content['tags'].unique()

array(['patchnotes', '', 'hide_store'], dtype=object)

In [47]:
os.makedirs("../data/raw", exist_ok=True)
df_content.to_csv("../data/raw/steam_content_batch1.csv", index=False)

print("Saved raw content batch.")

Saved raw content batch.


In [51]:
df_content_clean = df_content.copy()

df_content_clean.columns = df_content_clean.columns.str.strip().str.lower()
df_content_clean = df_content_clean.drop_duplicates(subset="app_id")

os.makedirs("../data/cleaned", exist_ok=True)
df_content_clean.to_csv("../data/cleaned/steam_content_clean.csv", index=False)

print("Saved cleaned content data.")

Saved cleaned content data.
